In [ ]:
# --- 1. 标准库导入 ---
import os
import time
import datetime

# --- 2. 第三方库导入 ---
import pandas as pd
import tushare as ts
import akshare as ak
import jieba
from dotenv import load_dotenv
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# --- 3. 全局配置 ---
load_dotenv()
token = os.getenv('TUSHARE_TOKEN')
ts.set_token(token)
pro = ts.pro_api()

# 设置路径变量，方便全局调用
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [ ]:
# --- 1. 配置测试股票池 ---
stock_pool = {
    "600519": "贵州茅台",
    "000858": "五粮液",
    "300750": "宁德时代",
    "002594": "比亚迪",
    "601318": "中国平安",
    "600036": "招商银行",
    "000333": "美的集团",
    "600900": "长江电力",
    "601088": "中国神华",
    "600436": "片仔癀"
}

raw_news_path = f"{RAW_DIR}/raw_news_pool.csv"
all_news_list = []

# --- 2. 抓取个股新闻 (东方财富接口) ---
for stock_code, stock_name in stock_pool.items():
    print(f"正在抓取: {stock_name} ({stock_code}) ...")
    try:
        news_df = ak.stock_news_em(symbol=stock_code)
        if not news_df.empty:
            news_df['股票代码'] = stock_code
            news_df['股票名称'] = stock_name
            all_news_list.append(news_df)
        time.sleep(2)  # 礼貌抓取，避免触发反爬
    except Exception as e:
        print(f"抓取 {stock_name} 失败: {e}")

# --- 3. 持久化新闻数据 ---
if all_news_list:
    final_raw_news = pd.concat(all_news_list, ignore_index=True)
    final_raw_news.to_csv(raw_news_path, index=False, encoding='utf-8-sig')
    
    final_raw_news['发布时间'] = pd.to_datetime(final_raw_news['发布时间'])
    print(f"批量抓取完成！共抓取 {len(final_raw_news)} 条数据。")
    print(f"时间范围: {final_raw_news['发布时间'].min()} 至 {final_raw_news['发布时间'].max()}")

正在抓取: 贵州茅台 (600519) ...
正在抓取: 五粮液 (000858) ...
正在抓取: 宁德时代 (300750) ...
正在抓取: 比亚迪 (002594) ...
正在抓取: 中国平安 (601318) ...
正在抓取: 招商银行 (600036) ...
正在抓取: 美的集团 (000333) ...
正在抓取: 长江电力 (600900) ...
正在抓取: 中国神华 (601088) ...
正在抓取: 片仔癀 (600436) ...
批量抓取完成！共抓取 100 条数据。
时间范围: 2026-03-29 21:50:16 至 2026-04-27 19:50:00


In [40]:
# --- 1. 动态计算时间窗口 ---
# 读取刚才保存的新闻数据
df_news_temp = pd.read_csv(raw_news_path)
df_news_temp['发布时间'] = pd.to_datetime(df_news_temp['发布时间'])

# 获取新闻中的最早日期
news_min_date = df_news_temp['发布时间'].min()

# 核心逻辑：将起始日期往前推 5 天，确保计算涨跌幅时有足够的基准数据
calc_start_date = (news_min_date - datetime.timedelta(days=5)).strftime('%Y%m%d')
# 截止日期设为今天
calc_end_date = datetime.datetime.now().strftime('%Y%m%d')

print(f"--- 自动时间窗口计算 ---")
print(f"新闻最早日期: {news_min_date.strftime('%Y-%m-%d')}")
print(f"股价拉取起点 (前推5天): {calc_start_date}")
print(f"股价拉取终点 (今日): {calc_end_date}")

# --- 2. 批量获取股价 (Tushare) ---
raw_price_path = f"{RAW_DIR}/raw_price_pool.csv"
all_prices_list = []

for stock_code, stock_name in stock_pool.items():
    # 转换 Tushare 代码格式
    ts_code = f"{stock_code}.SH" if stock_code.startswith('6') else f"{stock_code}.SZ"
        
    try:
        # 使用动态计算的日期窗口
        df_p = pro.daily(ts_code=ts_code, start_date=calc_start_date, end_date=calc_end_date)
        if not df_p.empty:
            all_prices_list.append(df_p)
        time.sleep(0.6) # 遵守 Tushare 积分限制频次
    except Exception as e:
        print(f"{stock_name} 获取失败: {e}")

# --- 3. 保存原始股价数据 ---
if all_prices_list:
    final_raw_price = pd.concat(all_prices_list, ignore_index=True)
    final_raw_price.to_csv(raw_price_path, index=False, encoding='utf-8-sig')
    print(f"股价数据已自动对齐并存入 raw 文件夹，共 {len(final_raw_price)} 行。")

--- 自动时间窗口计算 ---
新闻最早日期: 2026-03-29
股价拉取起点 (前推5天): 20260324
股价拉取终点 (今日): 20260427
股价数据已自动对齐并存入 raw 文件夹，共 240 行。


In [41]:
# --- 1. 读取原始数据 ---
df_news = pd.read_csv(f"{RAW_DIR}/raw_news_pool.csv")
df_price = pd.read_csv(f"{RAW_DIR}/raw_price_pool.csv")

# --- 2. 清洗新闻文本：构建日度舆情特征 ---
# 统一日期格式
df_news['date'] = pd.to_datetime(df_news['发布时间']).dt.strftime('%Y-%m-%d')

# 将同一只股票同一天的新闻拼接，形成日度文本特征
daily_news = df_news.groupby(['date', '股票代码'])['新闻标题'].apply(lambda x: ' 。 '.join(x)).reset_index()
daily_news['股票代码'] = daily_news['股票代码'].astype(str)

# --- 3. 清洗股价并制作“预测标签” ---
# Tushare 日期转为标准格式并处理代码后缀
df_price['date'] = pd.to_datetime(df_price['trade_date'].astype(str)).dt.strftime('%Y-%m-%d')
df_price['股票代码'] = df_price['ts_code'].str[:6].astype(str)

# 按股票和日期升序排列，确保 shift 操作逻辑正确
df_price = df_price.sort_values(['股票代码', 'date'], ascending=True)

# 【核心逻辑】：计算次日收益率方向 (Predictive Labeling)
# shift(-1) 将明天的涨跌幅拉到今天，用于监督学习训练
df_price['next_pct_chg'] = df_price.groupby('股票代码')['pct_chg'].shift(-1)

# 生成 Label: 1 代表明天涨，0 代表明天跌/平
df_price['label'] = (df_price['next_pct_chg'] > 0).astype(int)

# 剔除掉没有次日数据的行（即每只股票的最后一天数据）
df_price_clean = df_price.dropna(subset=['label']).copy()

# --- 4. 最终对齐：合并 X (文本) 与 Y (标签) ---
final_dataset = pd.merge(
    daily_news, 
    df_price_clean[['date', '股票代码', 'label']], 
    on=['date', '股票代码'], 
    how='inner'
)

# --- 5. 保存清洗后的数据集 ---
processed_path = f"{PROCESSED_DIR}/sestm_aligned_dataset.csv"
final_dataset.to_csv(processed_path, index=False, encoding='utf-8-sig')

print(f"数据清洗对齐完成！")
print(f"训练样本总数: {len(final_dataset)} 条 (对应 [新闻文本-次日涨跌] 样本对)")
print(final_dataset[['date', '股票代码', 'label']].head())

数据清洗对齐完成！
训练样本总数: 36 条 (对应 [新闻文本-次日涨跌] 样本对)
         date    股票代码  label
0  2026-04-01  600436      0
1  2026-04-03  600900      0
2  2026-04-09  600900      1
3  2026-04-10  600900      1
4  2026-04-13  600436      0


In [42]:
# --- 1. 文本分词与过滤 ---
print(f"正在对 {len(final_dataset)} 条样本进行分词处理...")

# 加载停用词（确保路径下有该文件，或增加空集合容错）
stop_path = "../resources/cn_stopwords.txt"
stopwords = set()
if os.path.exists(stop_path):
    with open(stop_path, 'r', encoding='utf-8') as f:
        stopwords = {line.strip() for line in f.readlines()}

def clean_cut(text):
    # 分词并过滤：去停用词、去单字（无意义词）、去纯数字
    words = jieba.cut(str(text))
    return " ".join([w for w in words if w not in stopwords and len(w) > 1 and not w.isdigit()])

final_dataset['cutted_news'] = final_dataset['新闻标题'].apply(clean_cut)

# --- 2. 向量化与模型训练 ---
vec = CountVectorizer()
X = vec.fit_transform(final_dataset['cutted_news'])
Y = final_dataset['label']

# 使用朴素贝叶斯进行监督学习
model = MultinomialNB()
model.fit(X, Y)

# --- 3. 提取关键词权重 (对应论文 3.4 节: Most Impactful Words) ---
feature_names = vec.get_feature_names_out()
# 计算 Log-Likelihood 差异：Log(P(词|涨)) - Log(P(词|跌))
# 差值越大，代表该词与“上涨”的关联度越高
log_prob_ratio = model.feature_log_prob_[1] - model.feature_log_prob_[0]

# 构建情绪词典 DataFrame
word_importance = pd.DataFrame({
    'word': feature_names,
    'weight': log_prob_ratio
}).sort_values(by='weight', ascending=False)

# --- 4. 结果输出与验证 ---
print("\n" + "="*30)
print("SESTM-Lite 情绪词典生成完毕！")
print("预测‘涨’关联度最高的词 (Positive Top 10):")
print(word_importance.head(10))

print("\n预测‘跌’关联度最高的词 (Negative Top 10):")
print(word_importance.tail(10))

# 计算拟合准确率
final_dataset['predicted_signal'] = model.predict(X)
acc = accuracy_score(Y, final_dataset['predicted_signal'])
print(f"\n训练集拟合准确率: {acc:.2%}")

正在对 36 条样本进行分词处理...

SESTM-Lite 情绪词典生成完毕！
预测‘涨’关联度最高的词 (Positive Top 10):
    word    weight
8     19  1.930899
150   电站  1.930899
76   发电量  1.930899
189   降至  1.930899
131   梯级  1.930899
65    六座  1.930899
114   所属  1.643217
60    以下  1.643217
87    境内  1.643217
78    古田  1.643217

预测‘跌’关联度最高的词 (Negative Top 10):
      word    weight
0       01 -1.247155
6    13941 -1.247155
163     茅台 -1.247155
132     概念 -1.247155
63      公布 -1.534837
80      合计 -1.534837
124     方案 -1.534837
66     净利润 -1.534837
67     净流入 -1.534837
136     派现 -2.163446

训练集拟合准确率: 88.89%
